In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [2]:
import pandas as pd

df=pd.read_csv(path+"/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.head()
df.isnull().sum()

,0
review,0
sentiment,0


In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df["review"] = df["review"].str.lower()

In [6]:
import re

In [7]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

In [8]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [9]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [10]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [11]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [14]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [15]:

from nltk.stem import PorterStemmer

In [16]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
y = df["sentiment"]

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [23]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [24]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [25]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [26]:
import torch.nn as nn
import torch.optim as optim

In [27]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [28]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [29]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.34008410573005676
epoch = 2/10 and loss = 0.43403393030166626
epoch = 3/10 and loss = 0.23428630828857422
epoch = 4/10 and loss = 0.1910562664270401
epoch = 5/10 and loss = 0.4641951024532318
epoch = 6/10 and loss = 0.2182350605726242
epoch = 7/10 and loss = 0.34618622064590454
epoch = 8/10 and loss = 0.31579023599624634
epoch = 9/10 and loss = 0.3073004484176636
epoch = 10/10 and loss = 0.12029775232076645


In [30]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.64081879600685
